# El Parser Avanzado

En este notebook se trabajará con el dataset limpio de Google Play Store para convertir datos de texto a formatos numéricos y de fecha que puedan utilizarse en análisis posteriores.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('playstore_limpio.csv')

df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9636 entries, 0 to 9635
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             9636 non-null   object 
 1   Category        9636 non-null   object 
 2   Rating          8180 non-null   float64
 3   Reviews         9636 non-null   int64  
 4   Size            9636 non-null   object 
 5   Installs        9636 non-null   int64  
 6   Type            9635 non-null   object 
 7   Price           9636 non-null   float64
 8   Content Rating  9636 non-null   object 
 9   Genres          9636 non-null   object 
 10  Last Updated    9636 non-null   object 
 11  Current Ver     9628 non-null   object 
 12  Android Ver     9634 non-null   object 
dtypes: float64(2), int64(2), object(9)
memory usage: 978.8+ KB


## Reto 1: La Trampa de las Unidades de Medida

La columna `Size` contiene valores en Megabytes, Kilobytes y también el texto `Varies with device`. Para poder analizarla, primero necesitamos convertir todos los tamaños a Megabytes y dejar la columna como numérica.

In [3]:
df['Size'].head(20)

0      19M
1      14M
2     8.7M
3      25M
4     2.8M
5     5.6M
6      19M
7      29M
8      33M
9     3.1M
10     28M
11     12M
12     20M
13     21M
14     37M
15    2.7M
16    5.5M
17     17M
18     39M
19     31M
Name: Size, dtype: object

In [4]:
def convertir_size(size):
    if size == 'Varies with device':
        return np.nan
    
    if size.endswith('M'):
        return float(size[:-1])
    
    if size.endswith('k'):
        return float(size[:-1]) / 1024

In [5]:
df['Size'] = df['Size'].apply(convertir_size)

In [6]:
df['Size'].head(20)

0     19.0
1     14.0
2      8.7
3     25.0
4      2.8
5      5.6
6     19.0
7     29.0
8     33.0
9      3.1
10    28.0
11    12.0
12    20.0
13    21.0
14    37.0
15     2.7
16     5.5
17    17.0
18    39.0
19    31.0
Name: Size, dtype: float64

In [7]:
df['Size'].dtype

dtype('float64')

In [8]:
promedio_size = df['Size'].mean()

print(f'Peso promedio de las aplicaciones: {promedio_size:.2f} MB')

Peso promedio de las aplicaciones: 20.42 MB


### Respuesta reto 1

El peso promedio de las aplicaciones en el dataset es de **20.42 MB**. Para obtenerlo, primero se convirtió toda la columna `Size` a Megabytes y después se calculó el promedio con `.mean()`.

## Reto 2: El Tipo de Dato Cronológico

La columna `Last Updated` contiene fechas guardadas como texto. Para poder analizarlas correctamente, se convertirán al tipo de dato `datetime` y después se extraerá solamente el año.

In [9]:
df['Last Updated'].head()

0     January 7, 2018
1    January 15, 2018
2      August 1, 2018
3        June 8, 2018
4       June 20, 2018
Name: Last Updated, dtype: object

In [10]:
df['Last Updated'] = pd.to_datetime(df['Last Updated'])

In [11]:
df['Last Updated'].dtype

dtype('<M8[ns]')

In [12]:
df['Year_Updated'] = df['Last Updated'].dt.year

In [13]:
df[['Last Updated', 'Year_Updated']].head()

,Last Updated,Year_Updated
0,2018-01-07,2018
1,2018-01-15,2018
2,2018-08-01,2018
3,2018-06-08,2018
4,2018-06-20,2018


In [14]:
df['Year_Updated'].value_counts()

Year_Updated
2018    6271
2017    1785
2016     779
2015     448
2014     203
2013     108
2012      26
2011      15
2010       1
Name: count, dtype: int64

### Respuesta reto 2

El año en el que se actualizó la mayor cantidad de aplicaciones fue **2018**, con **6271 aplicaciones**. Esto se obtuvo utilizando `value_counts()` sobre la columna `Year_Updated`.

## Reto 3: La Decisión Arquitectónica

Al convertir la columna `Size`, los valores `Varies with device` se transformaron en `NaN`. Primero se revisará cuántos registros quedaron sin tamaño para después decidir cómo tratarlos.

In [15]:
df['Size'].isna().sum()

np.int64(1227)

In [16]:
mediana_size = df['Size'].median()

print(f'Mediana del tamaño: {mediana_size:.2f} MB')

Mediana del tamaño: 12.00 MB


In [17]:
df['Size'] = df['Size'].fillna(mediana_size)

In [18]:
df['Size'].isna().sum()

np.int64(0)

### Respuesta reto 3

Se encontraron **1227 registros** sin tamaño después de convertir `Varies with device` a `NaN`.

Decidí rellenar estos valores utilizando la mediana del tamaño de las aplicaciones.
Esto permite conservar los registros y evitar perder información útil para el modelo.
Además, la mediana se ve menos afectada por valores extremos, por lo que es una opción más estable para la imputación.